Chargement & récupération

In [1]:
import os
import re
import pandas as pd
from sqlalchemy import URL, create_engine, text
from IPython.display import display

BATCH = os.getenv("DQ_BATCH", "L-2026-10-01")
SCHEMA = os.getenv("DQ_SCHEMA", "bronze")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", SCHEMA):
    raise ValueError(f"DQ_SCHEMA doit être un identifiant SQL simple : {SCHEMA!r}")

DATABASE_URL = os.getenv("DQ_DATABASE_URL")
if DATABASE_URL:
    ENGINE = create_engine(DATABASE_URL)
else:
    ENGINE = create_engine(URL.create(
        "mysql+pymysql",
        username=os.getenv("DQ_DB_USER", "root"),
        password=os.getenv("DQ_DB_PASSWORD", "root"),
        host=os.getenv("DQ_DB_HOST", "mariadb"),
        port=int(os.getenv("DQ_DB_PORT", "3306")),
    ))

def query(sql, **params):
    with ENGINE.connect() as connection:
        return pd.read_sql(text(sql), connection, params=params, coerce_float=False)

TABLE_CLIENT = os.getenv("DQ_TABLE_CLIENT", "crm_client")
TABLE_CONTRAT = os.getenv("DQ_TABLE_CONTRAT", "crm_contrat")
for nom, table in [("DQ_TABLE_CLIENT", TABLE_CLIENT), ("DQ_TABLE_CONTRAT", TABLE_CONTRAT)]:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", table):
        raise ValueError(f"{nom} doit être un identifiant SQL simple : {table!r}")

clients = query(
    f"SELECT * FROM `{SCHEMA}`.`{TABLE_CLIENT}` WHERE _batch_id = :batch",
    batch=BATCH,
)
contrats = query(
    f"SELECT * FROM `{SCHEMA}`.`{TABLE_CONTRAT}` WHERE _batch_id = :batch",
    batch=BATCH,
)

def collecter_echecs(rule_id, table_cible, donnees, condition):
    condition = pd.Series(condition, index=donnees.index).fillna(False).astype(bool)
    anomalies = donnees.loc[~condition].copy()
    anomalies["rule_id"] = rule_id
    anomalies["table_cible"] = table_cible
    return anomalies

echecs_par_regle = []

debut = pd.to_datetime(contrats["date_debut"], errors="coerce")
fin = pd.to_datetime(contrats["date_fin"], errors="coerce")
echecs_par_regle.append(collecter_echecs(
    "RQ-001", TABLE_CONTRAT, contrats, debut.notna() & fin.notna() & fin.gt(debut)
))

clients_uniques = clients.loc[
    clients["client_id"].notna() & ~clients["client_id"].duplicated(keep=False),
    ["client_id", "type_personne", "raison_sociale", "date_naissance"],
].copy()
contrats_avec_client = contrats.merge(
    clients_uniques,
    left_on="souscripteur_id",
    right_on="client_id",
    how="left",
)
raison_sociale = contrats_avec_client["raison_sociale"].astype("string").str.strip()
est_personne_physique = (
    contrats_avec_client["client_id"].notna()
    & contrats_avec_client["type_personne"].astype("string").str.strip().str.lower().eq("souscripteur")
    & (raison_sociale.isna() | raison_sociale.eq(""))
)
physiques = contrats_avec_client.loc[est_personne_physique].copy()
debut_physique = pd.to_datetime(physiques["date_debut"], errors="coerce")
naissance = pd.to_datetime(physiques["date_naissance"], errors="coerce")
evaluables_mask = debut_physique.notna() & naissance.notna()
evaluables = physiques.loc[evaluables_mask].copy()
debut_evaluable = debut_physique.loc[evaluables_mask]
naissance_evaluable = naissance.loc[evaluables_mask]
age = debut_evaluable.dt.year - naissance_evaluable.dt.year
age -= (
    (debut_evaluable.dt.month < naissance_evaluable.dt.month)
    | ((debut_evaluable.dt.month == naissance_evaluable.dt.month)
       & (debut_evaluable.dt.day < naissance_evaluable.dt.day))
).astype(int)
echecs_par_regle.append(collecter_echecs(
    "RQ-002", TABLE_CONTRAT, evaluables, age.ge(18)
))
print("Souscripteurs physiques identifiés :", len(physiques))
print("Souscripteurs non évaluables exclus de RQ-002 :", len(physiques) - len(evaluables))

type_normalise = clients["type_personne"].astype("string").str.strip().str.lower()
souscripteurs = clients.loc[type_normalise.eq("souscripteur")].copy()
telephone = souscripteurs["telephone"].astype("string")
echecs_par_regle.append(collecter_echecs(
    "RQ-003", TABLE_CLIENT, souscripteurs,
    telephone.notna() & telephone.str.strip().ne(""),
))

pays = clients["pays"].astype("string").str.strip().str.upper()
france = clients.loc[pays.isin(["FR", "FRANCE"])].copy()
code_postal = france["cp"].astype("string").str.strip()
echecs_par_regle.append(collecter_echecs(
    "RQ-005", TABLE_CLIENT, france, code_postal.str.fullmatch(r"\d{5}", na=False)
))

echecs_par_regle.append(collecter_echecs(
    "RQ-009", TABLE_CLIENT, clients,
    clients["type_personne"].isin(["prospect", "souscripteur", "assure"]),
))

duree = pd.to_numeric(contrats["duree_garantie_mois"], errors="coerce")
echecs_par_regle.append(collecter_echecs(
    "RQ-010", TABLE_CONTRAT, contrats, duree.isin([12, 24, 36, 60])
))

for table_cible, donnees, colonne_cle in [
    (TABLE_CLIENT, clients, "client_id"),
    (TABLE_CONTRAT, contrats, "contrat_id"),
]:
    cle = donnees[colonne_cle]
    echecs_par_regle.append(collecter_echecs(
        "RQ-020", table_cible, donnees,
        cle.notna() & ~cle.duplicated(keep=False),
    ))

echecs = pd.concat(echecs_par_regle, ignore_index=True, sort=False)
ordre_regles = ["RQ-001", "RQ-002", "RQ-003", "RQ-005", "RQ-009", "RQ-010", "RQ-020"]
groupes = {
    rule_id: echecs.loc[echecs["rule_id"].eq(rule_id)].sample(frac=1, random_state=20261001).reset_index(drop=True)
    for rule_id in ordre_regles
}
parts = []
for position in range(min(max((len(groupe) for groupe in groupes.values()), default=0), 20)):
    for rule_id in ordre_regles:
        if len(parts) >= 20:
            break
        if position < len(groupes[rule_id]):
            parts.append(groupes[rule_id].iloc[[position]])
echantillon = pd.concat(parts, ignore_index=True, sort=False) if parts else echecs.head(0).copy()
print("Lot analysé :", BATCH)
print("Occurrences d'échec :", len(echecs))
print("Échantillon courant :", len(echantillon), "lignes")
display(echantillon.head(1))

Souscripteurs physiques identifiés : 91875
Souscripteurs non évaluables exclus de RQ-002 : 0


Lot analysé : L-2026-10-01
Occurrences d'échec : 9374
Échantillon courant : 20 lignes


,bronze_row_id,contrat_id,souscripteur_id,assure_id,code_type,date_debut,date_fin,duree_garantie_mois,prime_annuelle,date_derniere_cotisation,...,date_naissance,nom,prenom,adresse,cp,ville,pays,telephone,email,date_creation
0,604268,CT0004268,S-9869,0,T11,2018-01-08,2017-12-29,12,1096.76,2019-01-02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
print('Lignes sélectionnées :', len(echantillon))
display(echantillon.head(1))

Lignes sélectionnées : 20


,bronze_row_id,contrat_id,souscripteur_id,assure_id,code_type,date_debut,date_fin,duree_garantie_mois,prime_annuelle,date_derniere_cotisation,...,date_naissance,nom,prenom,adresse,cp,ville,pays,telephone,email,date_creation
0,604268,CT0004268,S-9869,0,T11,2018-01-08,2017-12-29,12,1096.76,2019-01-02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
familles = {
    'RQ-001': ('Cohérence des dates', 'Date absente ou date de fin antérieure ou égale au début', 'Vérifier les dates et les corriger après validation'),
    'RQ-002': ("Validité de l'âge", 'Âge inférieur à 18 ans au début du contrat', "Vérifier l'éligibilité du souscripteur au début du contrat"),
    'RQ-003': ('Complétude du téléphone', 'Téléphone NULL, vide ou composé d’espaces', 'Vérifier la source et compléter si possible'),
    'RQ-005': ('Format du code postal', 'Code postal français mal formaté', 'Vérifier et corriger le code postal'),
    'RQ-009': ('Valeur hors domaine', 'Type de personne non autorisé', 'Vérifier et corriger la valeur'),
    'RQ-010': ('Durée de garantie', 'Durée hors liste autorisée', 'Vérifier la durée et la corriger si nécessaire'),
    'RQ-020': ('Unicité des clés métier', 'Identifiant client ou contrat nul ou dupliqué', 'Data Owner : vérifier la clé dans la source'),
}

def commentaire(rule_id, position):
    return familles.get(rule_id, ('À analyser', 'Cause à déterminer', 'Action à définir'))[position]

for donnees in (echecs, echantillon):
    donnees['famille_erreur'] = donnees['rule_id'].map(lambda rule_id: commentaire(rule_id, 0))
    donnees['cause_probable'] = donnees['rule_id'].map(lambda rule_id: commentaire(rule_id, 1))
    donnees['action_proposee'] = donnees['rule_id'].map(lambda rule_id: commentaire(rule_id, 2))

echantillon['cause_confirmee'] = ''
echantillon['commentaire_analyse'] = ''

display(echantillon.head(1))
# Constat individuel calculé ; distinct d’une cause ou confirmation humaine.
def constat_factuel(ligne):
    def valeur(col):
        v = ligne.get(col)
        if pd.isna(v): return "NULL"
        if isinstance(v, str) and v == "": return "vide"
        if isinstance(v, str) and v.strip() == "": return "blanc"
        return repr(v)
    regle = ligne["rule_id"]
    if regle == "RQ-001":
        debut = pd.to_datetime(ligne.get("date_debut"), errors="coerce")
        fin = pd.to_datetime(ligne.get("date_fin"), errors="coerce")
        motif = "; ".join(label for date, label in ((debut, "début absent/invalide"), (fin, "fin absente/invalide")) if pd.isna(date))
        if not motif:
            motif = "fin égale au début" if fin == debut else "fin antérieure au début"
        return f"date_debut={valeur('date_debut')}; date_fin={valeur('date_fin')}; {motif}"
    if regle == "RQ-002":
        naissance = pd.to_datetime(ligne.get("date_naissance"), errors="coerce")
        debut = pd.to_datetime(ligne.get("date_debut"), errors="coerce")
        age = (debut.year - naissance.year - ((debut.month, debut.day) < (naissance.month, naissance.day))) if pd.notna(naissance) and pd.notna(debut) else "non calculable"
        return f"naissance={valeur('date_naissance')}; début={valeur('date_debut')}; âge au début={age}"
    if regle == "RQ-003": return f"telephone={valeur('telephone')}"
    if regle == "RQ-005": return f"pays={valeur('pays')}; cp={valeur('cp')}; format français attendu: cinq chiffres"
    if regle == "RQ-009": return f"type_personne={valeur('type_personne')}; valeurs attendues: prospect/souscripteur/assure"
    if regle == "RQ-010": return f"duree_garantie_mois={valeur('duree_garantie_mois')}; valeurs attendues: 12/24/36/60"
    return f"clé client={valeur('client_id')}; contrat={valeur('contrat_id')}; clé absente ou répétée dans le lot"
for donnees in (echecs, echantillon):
    donnees["constat_factuel_automatique"] = donnees.apply(constat_factuel, axis=1)


,bronze_row_id,contrat_id,souscripteur_id,assure_id,code_type,date_debut,date_fin,duree_garantie_mois,prime_annuelle,date_derniere_cotisation,...,ville,pays,telephone,email,date_creation,famille_erreur,cause_probable,action_proposee,cause_confirmee,commentaire_analyse
0,604268,CT0004268,S-9869,0,T11,2018-01-08,2017-12-29,12,1096.76,2019-01-02,...,NaN,NaN,NaN,NaN,NaN,Cohérence des dates,Date absente ou date de fin antérieure ou égal...,Vérifier les dates et les corriger après valid...,,


In [4]:
bilan = (
    echantillon.groupby(
        ["rule_id", "table_cible", "famille_erreur", "cause_probable", "action_proposee"],
        as_index=False,
    )
    .size()
    .rename(columns={
        "rule_id": "Règle",
        "table_cible": "Table cible",
        "famille_erreur": "Famille d'erreur",
        "cause_probable": "Cause probable",
        "action_proposee": "Action proposée (à qui ?)",
        "size": "Lignes",
    })
    .sort_values("Règle")
)
bilan["Lignes (sur l'échantillon)"] = bilan["Lignes"].map(lambda nombre: f"{nombre}/{len(echantillon)}")
bilan = bilan[["Règle", "Table cible", "Famille d'erreur", "Lignes (sur l'échantillon)", "Cause probable", "Action proposée (à qui ?)"]]
display(bilan)

,Règle,Table cible,Famille d'erreur,Lignes (sur l'échantillon),Cause probable,Action proposée (à qui ?)
0,RQ-001,crm_contrat,Cohérence des dates,4/20,Date absente ou date de fin antérieure ou égal...,Vérifier les dates et les corriger après valid...
1,RQ-002,crm_contrat,Validité de l'âge,4/20,Âge inférieur à 18 ans au début du contrat,Vérifier l'éligibilité du souscripteur au débu...
2,RQ-003,crm_client,Complétude du téléphone,3/20,"Téléphone NULL, vide ou composé d’espaces",Vérifier la source et compléter si possible
3,RQ-005,crm_client,Format du code postal,3/20,Code postal français mal formaté,Vérifier et corriger le code postal
4,RQ-009,crm_client,Valeur hors domaine,3/20,Type de personne non autorisé,Vérifier et corriger la valeur
5,RQ-010,crm_contrat,Durée de garantie,3/20,Durée hors liste autorisée,Vérifier la durée et la corriger si nécessaire


In [5]:
from pathlib import Path
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pipelines/s5_engineering.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Racine du dépôt introuvable")
repertoire_preuves = ROOT / "livrables/preuves"
repertoire_preuves.mkdir(parents=True, exist_ok=True)
chemin_echecs = repertoire_preuves / "S2_US11_echecs_complets.csv"
chemin_echantillon = repertoire_preuves / "S2_US11_echantillon_commente.csv"
# Ne pas effacer une analyse humaine antérieure en régénérant l’échantillon.
if chemin_echantillon.exists():
    precedent = pd.read_csv(chemin_echantillon, sep=";", dtype=str, keep_default_na=False)
    if any(precedent.get(c, pd.Series(dtype=str)).str.strip().ne("").any()
           for c in ("cause_confirmee", "commentaire_analyse")):
        raise RuntimeError("Analyse humaine existante : rapprocher les annotations avant de remplacer cet export")
echecs.to_csv(chemin_echecs, sep=";", index=False, encoding="utf-8-sig")
echantillon.to_csv(chemin_echantillon, sep=";", index=False, encoding="utf-8-sig")
assert len(pd.read_csv(chemin_echecs, sep=";")) == len(echecs)
assert len(pd.read_csv(chemin_echantillon, sep=";")) == len(echantillon)
print(f"Export complet : {chemin_echecs} ({len(echecs)} occurrences)")
print(f"Échantillon commenté : {chemin_echantillon} ({len(echantillon)} lignes)")


Export complet : /tmp/assurconnect-audit-run/livrables/preuves/S2_US11_echecs_complets.csv (9374 occurrences)
Échantillon commenté : /tmp/assurconnect-audit-run/livrables/preuves/S2_US11_echantillon_commente.csv (20 lignes)
